In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

image_path = Path(
    r"E:\final-year-project\working-files\dataset\recognition-lfw\lfw_funneled\Aaron_Peirsol\Aaron_Peirsol_0001.jpg"
)

image = cv2.imread(str(image_path))

print(image_path.name)
print(image.shape)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
from app.config.config import (
    WEIGHTS_PATH,
    CONF,
    IOU,
    IMG_SIZE,
    DEVICE
)

from app.detection.face_detector import FaceDetector
from app.recognition.face_recognizer import FaceRecognizer

# Import directly from your registration script
from app.register.register_student import (
    augment_face,
    compute_mean_embedding
)

In [ ]:
detector = FaceDetector(
    WEIGHTS_PATH,
    device=DEVICE,
    conf=0.25,
    iou=0.5,
    img_size=320
)

recognizer = FaceRecognizer(device=DEVICE)

print("Models Loaded.")

In [ ]:
detections = detector.predict(image)

print(f"Detections: {len(detections)}")

for det in detections:
    print(det)

In [ ]:
output = image.copy()

for det in detections:

    x1 = int(det["x1"])
    y1 = int(det["y1"])
    x2 = int(det["x2"])
    y2 = int(det["y2"])

    conf = det["conf"]

    cv2.rectangle(
        output,
        (x1, y1),
        (x2, y2),
        (0,255,0),
        2
    )

    cv2.putText(
        output,
        f"{conf:.2f}",
        (x1,y1-5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,0),
        2
    )

plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
if len(detections):

    det = max(detections, key=lambda d: d["conf"])

    x1 = int(det["x1"])
    y1 = int(det["y1"])
    x2 = int(det["x2"])
    y2 = int(det["y2"])

    pad = 10

    x1 = max(0, x1-pad)
    y1 = max(0, y1-pad)
    x2 = min(image.shape[1], x2+pad)
    y2 = min(image.shape[0], y2+pad)

    face = image[y1:y2, x1:x2]

    face = cv2.resize(face, (112,112))

    plt.figure(figsize=(4,4))
    plt.imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

else:

    print("No face detected.")

In [ ]:
embedding = recognizer.get_embedding(face)

print("Embedding Shape :", embedding.shape)
print("Norm            :", np.linalg.norm(embedding))